# TrainWatcher Phase-02 Manual Test Lab (PyTorch)

This notebook is the **PyTorch** version of the Phase-02 manual validation suite.

It is designed for your environment where **PyTorch can use the GPU** while modern TensorFlow on native Windows cannot.

It validates:

- one-time email registration
- failure notification with cause
- rule-only interpretation
- hosted `hybrid` interpretation
- hosted `llm` mode with rule fallback
- best-model summary extraction
- optional BYOK fallback
- local summary text shown in the notebook for inspection

## Test order

Recommended manual order:

1. Run **email setup** once.
2. Run **failure scenario**.
3. Run **rule mode** scenario.
4. Run **hybrid mode** scenario.
5. Run **llm mode** scenario.
6. Run **best-model sweep** scenario.
7. Optionally run **BYOK fallback**.

In [1]:
# Uncomment if PyTorch / torchvision are not already installed in this kernel.
# !pip install trainwatcher torch torchvision

In [2]:
from trainwatcher import add_email

REGISTER_EMAIL = True
EMAIL_ADDRESS = "sambit.mahapatra.689@gmail.com"

# Run this only once, then set REGISTER_EMAIL = False again.
if REGISTER_EMAIL:
    add_email(EMAIL_ADDRESS)

Enter the 6-digit verification code sent to your email:  721127


In [3]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from trainwatcher import monitor, watch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Scenario toggles
RUN_FAILURE_SCENARIO = True
RUN_RULE_SCENARIO = True
RUN_HYBRID_SCENARIO = True
RUN_LLM_ONLY_SCENARIO = True
RUN_BEST_MODEL_SWEEP = True
RUN_BYOK_FALLBACK_SCENARIO = False

SEND_MANUAL_MILESTONES = False

# Optional advanced BYOK fallback only.
# os.environ["TRAINWATCHER_INTERPRETATION_FALLBACK"] = "byok"
# os.environ["TRAINWATCHER_LLM_API_KEY"] = "YOUR_OPENROUTER_KEY"
# os.environ["TRAINWATCHER_LLM_BASE_URL"] = "https://openrouter.ai/api/v1"
# os.environ["TRAINWATCHER_LLM_MODEL"] = "openai/gpt-oss-20b"

BASE_CONFIG = {
    "notifications": {"email": False, "telegram": False},
    "logging": {"enabled": False},
}

RULE_CONFIG = {
    **BASE_CONFIG,
    "interpretation": {
        "mode": "rule",
        "fallback": "rule",
        "hosted": {"enabled": True},
    },
}

HYBRID_CONFIG = {
    **BASE_CONFIG,
    "interpretation": {
        "mode": "hybrid",
        "fallback": "rule",
        "hosted": {"enabled": True},
    },
}

LLM_ONLY_CONFIG = {
    **BASE_CONFIG,
    "interpretation": {
        "mode": "llm",
        "fallback": "rule",
        "hosted": {"enabled": True},
    },
}

BYOK_FALLBACK_CONFIG = {
    **BASE_CONFIG,
    "interpretation": {
        "mode": "hybrid",
        "fallback": "byok",
        "hosted": {"enabled": True},
        "byok": {
            "api_key": os.getenv("TRAINWATCHER_LLM_API_KEY"),
            "base_url": os.getenv("TRAINWATCHER_LLM_BASE_URL"),
            "model": os.getenv("TRAINWATCHER_LLM_MODEL"),
            "max_tokens": 300,
            "temperature": 0.2,
            "timeout_seconds": 20,
        },
    },
}

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("BYOK configured:", bool(os.getenv("TRAINWATCHER_LLM_API_KEY")))

PyTorch version: 2.7.1+cu118
CUDA available: True
Device: cuda
BYOK configured: False


In [4]:
from IPython.display import Markdown, display

transform = transforms.Compose([
    transforms.ToTensor(),
])

full_train_dataset = datasets.FashionMNIST(root="./data", train=True, download=True, transform=transform)
full_test_dataset = datasets.FashionMNIST(root="./data", train=False, download=True, transform=transform)

train_indices = list(range(0, 12000))
val_indices = list(range(12000, 14000))
test_indices = list(range(0, 2000))

train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_train_dataset, val_indices)
test_dataset = Subset(full_test_dataset, test_indices)

print("Train subset:", len(train_dataset))
print("Validation subset:", len(val_dataset))
print("Test subset:", len(test_dataset))


def show_summary(snapshot, heading):
    print("=" * 80)
    print(heading)
    print("=" * 80)
    print(snapshot["summary"])
    display(Markdown("```text " + snapshot["summary"] + "```"))

Train subset: 12000
Validation subset: 2000
Test subset: 2000


In [5]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


def build_model(num_classes=10, dropout=0.3, target_device=None):
    target_device = target_device or device
    return SmallCNN(num_classes=num_classes, dropout=dropout).to(target_device)


def make_loaders(batch_size=128):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader


def evaluate_model(model, dataloader, criterion, run_device=None):
    run_device = run_device or device
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs = inputs.to(run_device)
            targets = targets.to(run_device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            total_loss += loss.item() * inputs.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == targets).sum().item()
            total_examples += inputs.size(0)
    return total_loss / total_examples, total_correct / total_examples


def train_epochs(model, train_loader, val_loader, epochs, learning_rate, scenario_name, run_device=None):
    run_device = run_device or device
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_examples = 0

        for inputs, targets in train_loader:
            inputs = inputs.to(run_device)
            targets = targets.to(run_device)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == targets).sum().item()
            total_examples += inputs.size(0)

        train_loss = total_loss / total_examples
        train_acc = total_correct / total_examples
        val_loss, val_acc = evaluate_model(model, val_loader, criterion, run_device=run_device)

        monitor.log(
            epoch=epoch,
            loss=float(train_loss),
            accuracy=float(train_acc),
            val_loss=float(val_loss),
            val_accuracy=float(val_acc),
            lr=float(optimizer.param_groups[0]["lr"]),
            scenario=scenario_name,
        )

    return model


class ManualSearchResult:
    def __init__(self, best_model, best_params, best_score, best_index):
        self.best_estimator_ = best_model
        self.best_params_ = best_params
        self.best_score_ = best_score
        self.best_index_ = best_index

## Scenario 1 — Deliberate failure

This creates the wrong output dimension (`5` classes instead of `10`).
To avoid poisoning the CUDA context, this scenario is forced onto CPU.
Expected result:

- failure email
- failure cause in summary
- notebook continues because the exception is handled locally

In [6]:
if RUN_FAILURE_SCENARIO:
    monitor.reset()
    monitor.start()
    try:
        if SEND_MANUAL_MILESTONES:
            monitor.notify("Failure scenario started: deliberate wrong output dimension")
        train_loader, val_loader, _ = make_loaders(batch_size=128)
        cpu_device = torch.device("cpu")
        bad_model = build_model(num_classes=5, dropout=0.3, target_device=cpu_device)
        train_epochs(
            bad_model,
            train_loader,
            val_loader,
            epochs=1,
            learning_rate=1e-3,
            scenario_name="failure_wrong_output_dim",
            run_device=cpu_device,
        )
        snapshot = monitor.end(config=RULE_CONFIG)
        show_summary(snapshot, "Unexpected success in failure scenario")
    except Exception as exc:
        snapshot = monitor.fail(exc, config=RULE_CONFIG)
        print("Expected failure captured:", exc)
        show_summary(snapshot, "Failure Scenario Summary")

Expected failure captured: Target 5 is out of bounds.
Failure Scenario Summary
Training Failed

Runtime: 0s
Error: Target 5 is out of bounds.

Likely Cause: The target labels exceed the model output dimension, which usually means the final layer class count does not match the dataset labels.

Suggestions:
- Set the final output dimension to match the dataset label count.
- Verify that target labels are in the expected range for the loss function.
- Check any class remapping or label encoding step before training.


```text Training Failed

Runtime: 0s
Error: Target 5 is out of bounds.

Likely Cause: The target labels exceed the model output dimension, which usually means the final layer class count does not match the dataset labels.

Suggestions:
- Set the final output dimension to match the dataset label count.
- Verify that target labels are in the expected range for the loss function.
- Check any class remapping or label encoding step before training.```

## Scenario 2 — Rule mode only

This uses a deliberately high learning rate and too many epochs for the subset.
Expected result:

- completion email
- rule observation such as divergence, plateau, or overfitting
- suggestions
- **no** hosted LLM section

In [7]:
if RUN_RULE_SCENARIO:
    monitor.reset()
    monitor.start()
    if SEND_MANUAL_MILESTONES:
        monitor.notify("Rule scenario started")

    train_loader, val_loader, _ = make_loaders(batch_size=128)
    rule_model = build_model(num_classes=10, dropout=0.1)
    train_epochs(
        rule_model,
        train_loader,
        val_loader,
        epochs=8,
        learning_rate=5e-2,
        scenario_name="rule_high_lr",
    )

    snapshot = monitor.end(config=RULE_CONFIG)
    show_summary(snapshot, "Rule Mode Summary")

Rule Mode Summary
Training Completed

Runtime: 28s
Best Validation Accuracy: 0.831
Epoch of Best Model: 7
Final Loss: 0.532


```text Training Completed

Runtime: 28s
Best Validation Accuracy: 0.831
Epoch of Best Model: 7
Final Loss: 0.532```

## Scenario 3 — Hosted hybrid mode

Expected result:

- completion email
- rule observation and suggestions
- hosted `LLM Interpretation` section

In [8]:
if RUN_HYBRID_SCENARIO:
    monitor.reset()
    monitor.start()
    if SEND_MANUAL_MILESTONES:
        monitor.notify("Hybrid scenario started")

    train_loader, val_loader, test_loader = make_loaders(batch_size=128)
    hybrid_model = build_model(num_classes=10, dropout=0.3)
    hybrid_model = train_epochs(
        hybrid_model,
        train_loader,
        val_loader,
        epochs=5,
        learning_rate=1e-3,
        scenario_name="hybrid_stable",
    )

    test_loss, test_acc = evaluate_model(hybrid_model, test_loader, nn.CrossEntropyLoss())
    print("Hybrid test accuracy:", round(float(test_acc), 4))

    snapshot = monitor.end(config=HYBRID_CONFIG)
    show_summary(snapshot, "Hybrid Mode Summary")

Hybrid test accuracy: 0.885
Hybrid Mode Summary
Training Completed

Runtime: 20s
Best Validation Accuracy: 0.8685
Epoch of Best Model: 5
Final Loss: 0.362

Observation: Training and validation loss both improved over the recent window.

Suggestions:
- Keep checkpointing the best validation model.
- Enable early stopping to avoid drifting past the optimum.
- Run a short hyperparameter sweep around the current settings.

LLM Interpretation:
The model has completed a 5-epoch hybrid training with a runtime of 20 seconds. It achieved a best validation accuracy of 0.8685 at epoch 5, indicating convergence. Both training and validation losses improved over the recent window, suggesting the model is generalizing well. 

Practical next steps:
1. Keep the best validation model checkpointed for future use.
2. Enable early stopping to prevent overfitting.
3. Run a short hyperparameter sweep around the current settings to further optimize the model.


```text Training Completed

Runtime: 20s
Best Validation Accuracy: 0.8685
Epoch of Best Model: 5
Final Loss: 0.362

Observation: Training and validation loss both improved over the recent window.

Suggestions:
- Keep checkpointing the best validation model.
- Enable early stopping to avoid drifting past the optimum.
- Run a short hyperparameter sweep around the current settings.

LLM Interpretation:
The model has completed a 5-epoch hybrid training with a runtime of 20 seconds. It achieved a best validation accuracy of 0.8685 at epoch 5, indicating convergence. Both training and validation losses improved over the recent window, suggesting the model is generalizing well. 

Practical next steps:
1. Keep the best validation model checkpointed for future use.
2. Enable early stopping to prevent overfitting.
3. Run a short hyperparameter sweep around the current settings to further optimize the model.```

## Scenario 4 — Hosted llm mode

Expected result:

- completion email
- hosted `LLM Interpretation` section
- if hosted path fails, rule-based summary still appears

In [9]:
if RUN_LLM_ONLY_SCENARIO:
    monitor.reset()
    monitor.start()
    if SEND_MANUAL_MILESTONES:
        monitor.notify("LLM-only scenario started")

    train_loader, val_loader, _ = make_loaders(batch_size=128)
    llm_model = build_model(num_classes=10, dropout=0.25)
    train_epochs(
        llm_model,
        train_loader,
        val_loader,
        epochs=4,
        learning_rate=1e-3,
        scenario_name="llm_only_stable",
    )

    snapshot = monitor.end(config=LLM_ONLY_CONFIG)
    show_summary(snapshot, "LLM Mode Summary")

LLM Mode Summary
Training Completed

Runtime: 15s
Best Validation Accuracy: 0.856
Epoch of Best Model: 4
Final Loss: 0.408

LLM Interpretation:
The model (LLM) has completed training in 15 seconds with 4 epochs. The validation accuracy peaked at 0.856 in the final epoch, indicating good convergence. Both training and validation losses decreased, suggesting the model is generalizing well. However, the training loss is still higher than the validation loss, indicating potential overfitting.

Practical next steps:

1. Keep the best validation model checkpointed for future use.
2. Enable early stopping to prevent overfitting.
3. Run a short hyperparameter sweep to fine-tune the current settings and potentially improve performance.


```text Training Completed

Runtime: 15s
Best Validation Accuracy: 0.856
Epoch of Best Model: 4
Final Loss: 0.408

LLM Interpretation:
The model (LLM) has completed training in 15 seconds with 4 epochs. The validation accuracy peaked at 0.856 in the final epoch, indicating good convergence. Both training and validation losses decreased, suggesting the model is generalizing well. However, the training loss is still higher than the validation loss, indicating potential overfitting.

Practical next steps:

1. Keep the best validation model checkpointed for future use.
2. Enable early stopping to prevent overfitting.
3. Run a short hyperparameter sweep to fine-tune the current settings and potentially improve performance.```

## Scenario 5 — Best-model sweep

This is a compact manual learning-rate sweep.
Expected result:

- completion email
- best-model section in summary
- best params + best score shown locally

In [10]:
if RUN_BEST_MODEL_SWEEP:
    monitor.reset()
    monitor.start()
    if SEND_MANUAL_MILESTONES:
        monitor.notify("Best-model sweep started")

    train_loader, val_loader, _ = make_loaders(batch_size=128)
    candidate_lrs = [1e-2, 1e-3, 5e-4]
    results = []

    for idx, lr in enumerate(candidate_lrs):
        model = build_model(num_classes=10, dropout=0.3)
        model = train_epochs(
            model,
            train_loader,
            val_loader,
            epochs=3,
            learning_rate=lr,
            scenario_name=f"sweep_lr_{lr}",
        )
        val_loss, val_acc = evaluate_model(model, val_loader, nn.CrossEntropyLoss())
        results.append((float(val_acc), lr, idx, model))
        monitor.log(epoch=idx + 1, val_accuracy=float(val_acc), lr=float(lr))

    best_score, best_lr, best_index, best_model = max(results, key=lambda item: item[0])
    search_result = ManualSearchResult(
        best_model=best_model,
        best_params={"learning_rate": best_lr, "epochs": 3, "batch_size": 128},
        best_score=best_score,
        best_index=best_index,
    )
    monitor.set_best_model(search_result)

    snapshot = monitor.end(config=HYBRID_CONFIG)
    show_summary(snapshot, "Best Model Sweep Summary")
    print("Best params:", search_result.best_params_)
    print("Best score:", round(search_result.best_score_, 4))

NotificationError: Failed to send notification.

## Scenario 6 — Optional BYOK fallback

Use this only if:

- you want to explicitly test BYOK fallback
- or your hosted interpretation quota has been exhausted

Expected result:

- hosted path is attempted first
- BYOK may provide `LLM Interpretation` if hosted path fails
- rule output remains the minimum backup

In [ ]:
if RUN_BYOK_FALLBACK_SCENARIO:
    monitor.reset()
    monitor.start()

    train_loader, val_loader, _ = make_loaders(batch_size=128)
    byok_model = build_model(num_classes=10, dropout=0.3)
    train_epochs(
        byok_model,
        train_loader,
        val_loader,
        epochs=4,
        learning_rate=1e-3,
        scenario_name="byok_fallback",
    )

    snapshot = monitor.end(config=BYOK_FALLBACK_CONFIG)
    show_summary(snapshot, "BYOK Fallback Summary")

## Optional one-call API check

This is a small extra check for the public `watch(...)` wrapper.

In [ ]:
def wrapped_train_demo():
    train_loader, val_loader, _ = make_loaders(batch_size=128)
    demo_model = build_model(num_classes=10, dropout=0.3)
    train_epochs(
        demo_model,
        train_loader,
        val_loader,
        epochs=2,
        learning_rate=1e-3,
        scenario_name="watch_wrapper_demo",
    )
    return demo_model

# Uncomment if you want to verify the one-call public API too.
# wrapped_model = watch(wrapped_train_demo, interpretation="hybrid", config=HYBRID_CONFIG)

## Manual verification checklist

For each scenario, verify both:

1. the **email content**
2. the **printed local summary** in the notebook

Checklist:

- failure summary includes the real error cause
- rule summary includes observation + suggestions but no LLM section
- hybrid summary includes hosted LLM interpretation
- llm summary includes hosted interpretation, or rule fallback if hosted path fails
- best-model summary includes model/score/params
- BYOK fallback only matters if you intentionally test it
- PyTorch uses GPU if `CUDA available: True`